!/usr/bin/env python
coding: utf-8

In [ ]:
# # Task 1 – Exploratory Data Analysis
# **10 Academy KAIM-9 | Week 3: End-to-End Insurance Risk Analytics**
#
# **Objective:** Load the ACIS historical claims dataset, assess its quality,
# and uncover initial patterns in risk and profitability through descriptive
# statistics and well-designed visualisations.
#
# **Dataset:** 18 months of car-insurance policy, client, vehicle, and claim
# data (Feb 2014 – Aug 2015).

## 0. Setup – Imports and Configuration

In [ ]:
# Standard library
import warnings
warnings.filterwarnings("ignore")

# Third-party
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Local source modules (from src/)
from src.data_loader import load_data, get_quality_report
from src.eda_utils import (
    plot_numerical_distributions,
    plot_categorical_distributions,
    plot_correlation_matrix,
    plot_scatter_premium_claims,
    plot_loss_ratio_by_group,
    plot_boxplots_outliers,
    plot_temporal_trends,
    plot_geographic_comparison,
    summarise_loss_ratio,
)

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

## 1. Load Data

`load_data` reads the pipe-delimited CSV, enforces correct dtypes for all
numeric, categorical, and date columns, and appends `LossRatio` and `Margin`.

In [ ]:
DATA_PATH = "data/insurance_data.csv"

df = load_data(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

## 2. Data Summarization

### 2a. Descriptive statistics for numerical features

The `.describe()` call below surfaces the central tendency, spread, and
extreme values for all numeric columns.  Pay particular attention to
`TotalClaims` (right-skewed distributions are common in insurance) and
`LossRatio` (values > 1 indicate unprofitable policies).

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numerical_cols].describe().T.round(2)

# ### 2b. Data types review
#
# Confirm that:
# - Financial columns are `float64` (not `object`).
# - Grouping columns are `category` (saves memory and speeds groupby).
# - Date columns are `datetime64`.

df.dtypes.to_frame("dtype").reset_index().rename(columns={"index": "column"})

## 3. Data Quality Assessment

`get_quality_report` returns a DataFrame sorted by `missing_pct` so the
most problematic columns appear first.  Columns with > 30 % missing values
will be considered for removal; columns with < 5 % missing values will be
imputed during modeling.

In [ ]:
quality_report = get_quality_report(df)
print("Columns with ANY missing values:")
quality_report[quality_report["missing_count"] > 0]

# Check for fully duplicate rows (same PolicyID and TransactionMonth).
n_duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {n_duplicates:,}")

## 4. Univariate Analysis

In [ ]:
# ### 4a. Numerical distributions
#
# Histograms with KDE overlays reveal skewness in financial variables.
# `TotalPremium` and `TotalClaims` are expected to be right-skewed — a log
# transformation may be necessary before linear modeling.

fig_num = plot_numerical_distributions(
    df,
    cols=["TotalPremium", "TotalClaims", "SumInsured",
          "CustomValueEstimate", "LossRatio", "Margin"],
)
plt.show()

# ### 4b. Categorical distributions
#
# Bar charts for the most frequent categories across client, location, and
# vehicle attributes.  Imbalanced categories (e.g. one dominant province)
# should inform stratified sampling in later tasks.

fig_cat = plot_categorical_distributions(
    df,
    cols=["Province", "Gender", "VehicleType", "CoverType", "Make"],
    top_n=10,
)
plt.show()

## 5. Bivariate / Multivariate Analysis

In [ ]:
# ### 5a. Correlation matrix
#
# The Pearson correlation matrix highlights linear relationships between
# numeric features.  Strong correlations with `TotalClaims` are valuable
# inputs for the predictive model in Task 4.

fig_corr = plot_correlation_matrix(
    df,
    cols=["TotalPremium", "TotalClaims", "SumInsured",
          "CustomValueEstimate", "LossRatio", "Margin"],
)
plt.show()

# ### 5b. TotalPremium vs TotalClaims by Province
#
# Each point is one policy.  Points above the 45-degree break-even line
# (red dashes) represent policies where claims exceeded the premium collected
# — these are loss-making.  Clustering by Province reveals whether certain
# regions systematically generate higher-than-expected claims.

fig_scatter = plot_scatter_premium_claims(df, hue_col="Province", sample_n=8000)
plt.show()

# ### 5c. TotalPremium vs TotalClaims by ZipCode (PostalCode)
#
# Same chart but coloured by PostalCode to explore sub-provincial variation.

fig_scatter_zip = plot_scatter_premium_claims(df, hue_col="PostalCode", sample_n=8000)
plt.show()

## 6. Geographic Trends

Compare Loss Ratio across provinces.  Red bars exceed the portfolio
average (dashed line), indicating regions where ACIS is under-pricing risk.

In [ ]:
fig_geo = plot_geographic_comparison(df, kpi_col="LossRatio")
plt.show()

# Tabular summary — useful for the final report.
geo_summary = summarise_loss_ratio(df, group_col="Province")
print(geo_summary.to_string(index=False))

## 7. Outlier Detection

Box plots use the IQR rule to flag extreme values.  Policies with
`TotalClaims` far above the upper whisker likely represent large accidents
or fraud and may need separate treatment (e.g. winsorisation or a separate
high-severity model).

In [ ]:
fig_box = plot_boxplots_outliers(df)
plt.show()

# Quantify outliers using the 1.5×IQR rule for key financial columns.
for col in ["TotalPremium", "TotalClaims", "CustomValueEstimate"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"{col}: {n_out:,} outliers ({n_out / len(df):.1%} of policies)")

## 8. Guiding Questions

In [ ]:
# ### Q1. Overall Loss Ratio — by Province, VehicleType, Gender
print("=== Loss Ratio by Province ===")
print(summarise_loss_ratio(df, "Province")[
    ["Province", "policy_count", "portfolio_loss_ratio"]
].to_string(index=False))

print("\n=== Loss Ratio by VehicleType ===")
print(summarise_loss_ratio(df, "VehicleType")[
    ["VehicleType", "policy_count", "portfolio_loss_ratio"]
].to_string(index=False))

print("\n=== Loss Ratio by Gender ===")
print(summarise_loss_ratio(df, "Gender")[
    ["Gender", "policy_count", "portfolio_loss_ratio"]
].to_string(index=False))

# ### Q2. Outliers in TotalClaims and CustomValueEstimate
# (Already computed in Section 7 above.)

# ### Q3. Temporal trends in claim frequency and severity
fig_temporal = plot_temporal_trends(df)
plt.show()

# ### Q4. Vehicle makes with highest and lowest claim amounts
make_claims = (
    df.groupby("Make", observed=True)["TotalClaims"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "avg_claim", "count": "policy_count"})
    .query("policy_count >= 50")           # ignore very rare makes
    .sort_values("avg_claim", ascending=False)
)
print("Top 10 makes by average claim amount:")
print(make_claims.head(10).to_string())
print("\nBottom 10 makes by average claim amount:")
print(make_claims.tail(10).to_string())

## 9. Creative Insight Visualisations

In [ ]:
# ### Plot A — Loss Ratio heatmap: Province × VehicleType
#
# A pivot table rendered as a heatmap reveals compound risk segments.
# For example, commercial vehicles in Gauteng may carry a fundamentally
# different risk profile than private passenger vehicles in the Western Cape.

pivot = (
    df.groupby(["Province", "VehicleType"], observed=True)["LossRatio"]
    .mean()
    .unstack("VehicleType")
)
fig_heatmap, ax = plt.subplots(figsize=(12, 6), dpi=120)
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn_r",
            linewidths=0.4, center=1.0, ax=ax)
ax.set_title("Mean Loss Ratio: Province × Vehicle Type\n"
             "(red = unprofitable, green = profitable)", fontweight="bold")
plt.tight_layout()
plt.show()

# ### Plot B — Premium distribution by Cover Type (violin)
#
# Violin plots combine the information of a box plot with a KDE, showing
# whether premium distributions within each cover type are unimodal or
# have multiple peaks (which might indicate hidden sub-segments).

fig_violin, ax = plt.subplots(figsize=(12, 5), dpi=120)
cover_order = (
    df.groupby("CoverType", observed=True)["TotalPremium"]
    .median()
    .sort_values(ascending=False)
    .index[:8]
)
sns.violinplot(
    data=df[df["CoverType"].isin(cover_order)],
    x="CoverType",
    y="TotalPremium",
    order=cover_order,
    palette="muted",
    inner="quartile",
    ax=ax,
)
ax.set_title("Premium Distribution by Cover Type", fontweight="bold")
ax.set_xlabel("Cover Type")
ax.set_ylabel("Total Premium (ZAR)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

# ### Plot C — Monthly claim frequency vs severity (dual-axis trend)
#
# Already rendered in Section 8 (Q3), reproduced here as a standalone
# creative insight chart for the final report.

fig_trend = plot_temporal_trends(df)
plt.show()

print("\nTask 1 EDA complete.")